# Kapitel 11: Den Transformer bauen

> „Wir sind, was wir wiederholt tun. Exzellenz ist daher keine Handlung, sondern eine Gewohnheit."
> — **Aristoteles**, Philosoph

---

## Was du lernen wirst

- Wie man Transformer-Blöcke zu einem vollständigen Sprachmodell stapelt
- Das Konfigurationsmuster, das Modellexperimente einfach macht
- Was der Language-Modeling-Head macht und warum wir ihn brauchen
- Weight Tying: Der elegante Trick, der 38 Millionen Parameter spart
- Wesentliche Sanity Checks zur Überprüfung deines Modells vor dem Training
- Wie man vortrainierte GPT-2-Gewichte in deine Architektur lädt

---

## Setup

Installieren wir zunächst die erforderlichen Pakete:

In [ ]:
# Erforderliche Pakete installieren
!pip install -q torch transformers

In [ ]:
# ===== IMPORTS =====
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
from transformers import AutoTokenizer

# Gerät festlegen
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Verwendetes Gerät: {device}")

## 1. Komponenten aus Kapitel 10

Holen wir zunächst die Komponenten, die wir in Kapitel 10 gebaut haben: `MultiHeadAttention`, `FeedForward` und `TransformerBlock`.

In [ ]:
# ===== MULTI-HEAD ATTENTION (aus Kapitel 10) =====

class MultiHeadAttention(nn.Module):
    """Effiziente mehrköpfige Aufmerksamkeit (bündelt alle Köpfe zusammen)."""

    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model muss durch num_heads teilbar sein"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        # Kombinierte QKV-Projektion
        # bias=True entspricht GPT-2, damit wir später dessen vortrainierte Biases laden können
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        # Projektion zu Q, K, V
        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        # Skalierte Punktprodukt-Aufmerksamkeit
        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)
        scores = scores / math.sqrt(d_k)

        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Gewichtete Summe und verketten
        attn_output = attn_weights @ V
        attn_output = attn_output.transpose(1, 2)
        attn_output = attn_output.reshape(batch, seq, d_model)

        output = self.out_proj(attn_output)
        return output, attn_weights

print("MultiHeadAttention definiert!")

In [ ]:
# ===== FEEDFORWARD-NETZWERK (aus Kapitel 10) =====

class FeedForward(nn.Module):
    """Positionsweises Feedforward-Netzwerk."""

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x, approximate="tanh")  # GPT-2s GELU-Variante
        x = self.dropout(x)
        x = self.fc2(x)
        return x

print("FeedForward definiert!")

In [ ]:
# ===== TRANSFORMER-BLOCK (aus Kapitel 10) =====

class TransformerBlock(nn.Module):
    """Vollständiger Transformer-Block (Pre-Norm-Stil wie GPT-2)."""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Aufmerksamkeit mit residualer Verbindung
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)

        # FFN mit residualer Verbindung
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)

        return x, attn_weights

print("TransformerBlock definiert!")

## 2. Modellkonfiguration

Erstellen wir eine Konfigurationsdataclass, um alle Modellhyperparameter zu bündeln.

In [ ]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    """Konfiguration für das MiniGPT-Modell."""
    vocab_size: int = 50257      # GPT-2-Vokabulargrösse
    max_seq_len: int = 1024      # Maximale Kontextlänge
    embed_dim: int = 768         # Einbettungsdimension
    num_heads: int = 12          # Anzahl der Aufmerksamkeitsköpfe
    num_layers: int = 12         # Anzahl der Transformer-Blöcke
    d_ff: int = 3072             # Versteckte Dimension des Feedforward-Netzwerks
    dropout: float = 0.1         # Dropout-Wahrscheinlichkeit

    def __post_init__(self):
        """Konfiguration validieren."""
        assert self.embed_dim % self.num_heads == 0, \
            f"embed_dim ({self.embed_dim}) muss durch num_heads ({self.num_heads}) teilbar sein"


## 3. Das vollständige MiniGPT-Modell

Bauen wir nun das vollständige Modell, indem wir Transformer-Blöcke stapeln und den Language-Modeling-Head mit Weight Tying hinzufügen.

In [ ]:
class MiniGPT(nn.Module):
    """
    Ein minimales Sprachmodell im GPT-Stil.
    Kombiniert Einbettungen, Transformer-Blöcke und Language-Modeling-Head.
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # ===== Token- und Positionseinbettungen =====
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # ===== Transformer-Blöcke =====
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # ===== Finale Schichtnormalisierung =====
        self.ln_f = nn.LayerNorm(config.embed_dim)

        # ===== Language-Modeling-Head =====
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # ===== Weight Tying =====
        self.lm_head.weight = self.token_embed.weight

        # Gewichte initialisieren
        self._init_weights()

    def _init_weights(self):
        """Gewichte mit kleinen Zufallswerten initialisieren."""
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        """
        Vorwärtsdurchlauf durch das Modell.

        Args:
            token_ids: Eingabe-Token-IDs (batch, seq)
            return_attention: Ob Aufmerksamkeitsgewichte zurückgegeben werden sollen

        Returns:
            logits: Vokabularscores (batch, seq, vocab_size)
            attention_weights: Optionale Liste der Aufmerksamkeitsgewichte pro Block
        """
        batch, seq = token_ids.shape
        device = token_ids.device

        # Einbettungen
        tok_emb = self.token_embed(token_ids)
        positions = torch.arange(seq, device=device)
        pos_emb = self.pos_embed(positions)
        x = self.dropout(tok_emb + pos_emb)

        # Kausale Maske
        mask = torch.tril(torch.ones(seq, seq, device=device))

        # Transformer-Blöcke
        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        # Finale Normalisierung und Projektion
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if return_attention:
            return logits, attention_weights
        return logits


    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0,
                 do_sample=True, eos_token_id=None):
        """
        Tokens generieren, die input_ids fortsetzen (Prompt ist in der Ausgabe enthalten).

        Args:
            input_ids: Prompt-Token-IDs (batch, seq)
            max_new_tokens: Wie viele neue Tokens generiert werden
            temperature: Sampling-Temperatur (nur verwendet, wenn do_sample=True)
            do_sample: Aus der Verteilung samplen; False = gieriges argmax
            eos_token_id: Früh stoppen, wenn jede Sequenz dieses Token generiert

        Returns:
            Token-IDs (batch, seq + neue Tokens)
        """
        self.eval()

        # Positions-Einbettungen decken nur max_seq_len Positionen ab
        limit = self.config.max_seq_len
        if input_ids.shape[1] + max_new_tokens > limit:
            raise ValueError(
                f"Prompt ({input_ids.shape[1]} Tokens) + max_new_tokens "
                f"({max_new_tokens}) überschreitet den {limit}-Token-Kontext "
                "des Modells. Kürze den Prompt oder senke max_new_tokens.")
        for _ in range(max_new_tokens):
            logits = self(input_ids)
            next_logits = logits[:, -1, :]  # (batch, vocab_size)

            if do_sample:
                probs = F.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = next_logits.argmax(dim=-1, keepdim=True)

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_token_id is not None and (next_token == eos_token_id).all():
                break

        return input_ids


print("MiniGPT-Klasse definiert!")

In [ ]:
# Schnelltest mit kleiner Konfiguration
tiny_config = GPTConfig(embed_dim=64, num_heads=2, num_layers=2, d_ff=256)
model = MiniGPT(tiny_config)

# Vorwärtsdurchlauf testen
test_tokens = torch.randint(0, tiny_config.vocab_size, (2, 16))
logits = model(test_tokens)

print(f"Eingabeform: {test_tokens.shape}")
print(f"Ausgabeform: {logits.shape}")
print(f"\nErwartet: (2, 16, {tiny_config.vocab_size})")

## 4. Sanity Checks

Überprüfen wir mit 5 wesentlichen Tests, ob unser Modell korrekt verdrahtet ist.

In [ ]:
# ===== TEST 1: Form-Überprüfung =====

def test_forward_shapes(config):
    """Modellausgabeformen überprüfen."""
    model = MiniGPT(config)

    batch_size, seq_len = 2, 16
    token_ids = torch.randint(0, config.vocab_size, (batch_size, seq_len))

    logits = model(token_ids)

    expected_shape = (batch_size, seq_len, config.vocab_size)
    assert logits.shape == expected_shape, \
        f"Erwartet {expected_shape}, erhalten {logits.shape}"

    print(f"Form-Check BESTANDEN: {logits.shape}")

test_forward_shapes(tiny_config)

In [ ]:
# ===== TEST 2: Parameteranzahl =====
def count_parameters(model):
    """Gesamtzahl trainierbarer Parameter zählen."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def test_parameter_count(config):
    """Überprüfen, ob die Parameteranzahl angemessen ist."""
    model = MiniGPT(config)
    total = count_parameters(model)

    # GPT-2 Small hat ~124M Parameter
    print(f"Gesamtzahl Parameter: {total:,}")

    # Exakte Aufschlüsselung (einschliesslich Bias-Terme)
    embed_params = config.vocab_size * config.embed_dim  # Token-Einbettungen
    pos_params = config.max_seq_len * config.embed_dim   # Positionseinbettungen
    d = config.embed_dim
    block_params = config.num_layers * (
        4 * d * d + 4 * d +                     # Attention: QKV + Ausgabe, mit Biases
        2 * d * config.d_ff + config.d_ff + d + # FFN: fc1 + fc2, mit Biases
        4 * d                                   # LayerNorms
    )
    ln_f_params = 2 * d                          # Finale LayerNorm
    # Hinweis: lm_head-Parameter sind mit token_embed geteilt, werden also nicht separat gezählt

    expected = embed_params + pos_params + block_params + ln_f_params
    print(f"Erwartet: {expected:,}")
    assert total == expected, f"Anzahl stimmt nicht: {total:,} vs {expected:,}"


# Test mit kleiner Konfiguration
print("Kleines Modell:")
test_parameter_count(tiny_config)

# Test mit voller GPT-2-Konfiguration
print("\nGPT-2 Small:")
test_parameter_count(GPTConfig())


In [ ]:
# ===== TEST 3: Kausale Maskierung =====

def test_causal_masking():
    """Überprüfen, dass das Modell keine zukünftigen Tokens sehen kann."""
    config = GPTConfig(
        num_layers=1,
        embed_dim=64,
        num_heads=2,
        d_ff=256,
        dropout=0.0  # Kein Dropout für Determinismus
    )
    model = MiniGPT(config)
    model.eval()

    # Gleicher Präfix, unterschiedlicher Suffix
    tokens_a = torch.tensor([[100, 200, 300, 400]])
    tokens_b = torch.tensor([[100, 200, 300, 999]])  # Letztes Token unterschiedlich

    with torch.no_grad():
        logits_a = model(tokens_a)
        logits_b = model(tokens_b)

    # Positionen 0, 1, 2 sollten IDENTISCH sein
    for pos in range(3):
        assert torch.allclose(logits_a[0, pos], logits_b[0, pos], atol=1e-5), \
            f"Position {pos} Logits unterscheiden sich!"

    # Position 3 SOLLTE sich unterscheiden
    assert not torch.allclose(logits_a[0, 3], logits_b[0, 3], atol=1e-5), \
        "Position 3 Logits gleich trotz unterschiedlicher Eingabe!"

    print("Kausale Maskierung BESTANDEN!")

test_causal_masking()

In [ ]:
# ===== TEST 4: Weight Tying =====

def test_weight_tying():
    """Überprüfen, dass Einbettung und lm_head Gewichte teilen."""
    config = GPTConfig(embed_dim=64, num_heads=2, num_layers=2, d_ff=256)
    model = MiniGPT(config)

    # Sollte der GLEICHE Tensor sein
    assert model.lm_head.weight is model.token_embed.weight, \
        "Weight Tying fehlgeschlagen: unterschiedliche Tensoren!"

    # Einen ändern, prüfen ob der andere sich ändert
    with torch.no_grad():
        original = model.token_embed.weight[0, 0].item()
        model.token_embed.weight[0, 0] = 999.0

        assert model.lm_head.weight[0, 0].item() == 999.0, \
            "Weight Tying fehlgeschlagen: Änderungen werden nicht übertragen!"

        model.token_embed.weight[0, 0] = original

    print("Weight Tying BESTANDEN!")

test_weight_tying()

In [ ]:
# ===== TEST 5: Gradientenfluss =====

def test_gradient_flow():
    """Überprüfen, dass Gradienten alle Parameter erreichen."""
    config = GPTConfig(num_layers=2, embed_dim=64, num_heads=2, d_ff=256)
    model = MiniGPT(config)

    # Vorwärtsdurchlauf
    tokens = torch.randint(0, config.vocab_size, (1, 8))
    logits = model(tokens)

    # Rückwärtsdurchlauf
    loss = logits.sum()
    loss.backward()

    # Überprüfen, dass alle Parameter Gradienten haben
    for name, param in model.named_parameters():
        assert param.grad is not None, f"Kein Gradient für {name}"

    print("Gradientenfluss BESTANDEN!")

test_gradient_flow()

## 5. Dein erster Vorwärtsdurchlauf

Lass uns echten Text durch unser Modell laufen und sehen, was es mit zufälligen Gewichten ausgibt.

In [ ]:
# Tokenizer laden
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Kleines Modell erstellen
config = GPTConfig(num_layers=2, embed_dim=256, num_heads=4, d_ff=1024)
model = MiniGPT(config)
model.eval()

# Einen Prompt tokenisieren
prompt = "The quick brown fox"
token_ids = tokenizer.encode(prompt, return_tensors="pt")

print(f"Prompt: '{prompt}'")
print(f"Token-IDs: {token_ids}")
print(f"Tokens: {[tokenizer.decode([t]) for t in token_ids[0]]}")

In [ ]:
# Vorwärtsdurchlauf
with torch.no_grad():
    logits = model(token_ids)

print(f"Logits-Form: {logits.shape}")

# Top-5-Vorhersagen für das nächste Token abrufen
last_logits = logits[0, -1, :]
top_probs, top_indices = torch.softmax(last_logits, dim=-1).topk(5)

print(f"\nTop-5-Vorhersagen nach '{prompt}':")
for prob, idx in zip(top_probs, top_indices):
    token = tokenizer.decode([idx])
    print(f"  '{token}': {prob:.4f}")

print("\n(Zufällige Vorhersagen - Modell hat zufällige Gewichte!)")

## 6. Vortrainierte Gewichte laden

Jetzt der spannende Teil: Laden wir echte GPT-2-Gewichte in unser MiniGPT!

In [ ]:
def load_gpt2_weights(model, model_name="gpt2"):
    """
    Vortrainierte GPT-2-Gewichte in unser MiniGPT-Modell laden.
    """
    from transformers import GPT2LMHeadModel

    print(f"Lade Gewichte von '{model_name}'...")

    # HuggingFace-Modell laden
    hf_model = GPT2LMHeadModel.from_pretrained(model_name)
    hf_state = hf_model.state_dict()

    # State Dict unseres Modells
    our_state = model.state_dict()

    # Einbettungen kopieren
    our_state['token_embed.weight'].copy_(hf_state['transformer.wte.weight'])
    our_state['pos_embed.weight'].copy_(hf_state['transformer.wpe.weight'])

    # Jeden Transformer-Block kopieren
    for i in range(model.config.num_layers):
        # Schichtnormalisierungen
        our_state[f'blocks.{i}.ln1.weight'].copy_(
            hf_state[f'transformer.h.{i}.ln_1.weight'])
        our_state[f'blocks.{i}.ln1.bias'].copy_(
            hf_state[f'transformer.h.{i}.ln_1.bias'])
        our_state[f'blocks.{i}.ln2.weight'].copy_(
            hf_state[f'transformer.h.{i}.ln_2.weight'])
        our_state[f'blocks.{i}.ln2.bias'].copy_(
            hf_state[f'transformer.h.{i}.ln_2.bias'])

        # Aufmerksamkeit (Gewichte müssen transponiert werden; Biases direkt kopieren)
        our_state[f'blocks.{i}.attn.qkv_proj.weight'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_attn.weight'].T)
        our_state[f'blocks.{i}.attn.qkv_proj.bias'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_attn.bias'])
        our_state[f'blocks.{i}.attn.out_proj.weight'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_proj.weight'].T)
        our_state[f'blocks.{i}.attn.out_proj.bias'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_proj.bias'])

        # FFN (muss transponiert werden!)
        our_state[f'blocks.{i}.ffn.fc1.weight'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_fc.weight'].T)
        our_state[f'blocks.{i}.ffn.fc1.bias'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_fc.bias'])
        our_state[f'blocks.{i}.ffn.fc2.weight'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_proj.weight'].T)
        our_state[f'blocks.{i}.ffn.fc2.bias'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_proj.bias'])

    # Finale Schichtnormalisierung
    our_state['ln_f.weight'].copy_(hf_state['transformer.ln_f.weight'])
    our_state['ln_f.bias'].copy_(hf_state['transformer.ln_f.bias'])

    # Hinweis: lm_head.weight ist mit token_embed.weight geteilt, bereits kopiert

    print("Gewichte erfolgreich geladen!")
    return model

In [ ]:
@torch.no_grad()
def generate_simple(model, tokenizer, prompt, max_new_tokens=20):
    """
    Text mit gierigem Dekodieren generieren (immer das wahrscheinlichste Token wählen).

    Args:
        model: Unser MiniGPT mit geladenen Gewichten
        tokenizer: GPT-2-Tokenizer
        prompt: Starttext
        max_new_tokens: Wie viele neue Tokens generiert werden

    Returns:
        Generierter Text als String
    """
    device = next(model.parameters()).device
    token_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # do_sample=False = gierig: argmax bei jedem Schritt
    output_ids = model.generate(
        token_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Zurück zu Text dekodieren (Prompt enthalten)
    return tokenizer.decode(output_ids[0])


In [ ]:
# Modell mit GPT-2-Small-Konfiguration erstellen
config = GPTConfig()  # Standardwerte entsprechen GPT-2 Small
model = MiniGPT(config)

# Vortrainierte Gewichte laden
model = load_gpt2_weights(model, "gpt2")
model = model.to(device)

print(f"\nModell auf Gerät: {device}")
print(f"Parameter: {count_parameters(model):,}")

In [ ]:
# ===== TEST 6: Logits-Parität mit HuggingFace GPT-2 =====
def test_gpt2_parity(model, tokenizer):
    """Überprüfen, dass unser geladenes Modell die HuggingFace-GPT-2-Logits reproduziert."""
    from transformers import GPT2LMHeadModel

    hf_model = GPT2LMHeadModel.from_pretrained("gpt2").eval()
    model.eval()

    # Auf dem Gerät des Modells vergleichen (HF-Modell folgt dorthin)
    device = next(model.parameters()).device
    hf_model = hf_model.to(device)
    ids = tokenizer.encode("The quick brown fox jumps over",
                           return_tensors="pt").to(device)
    with torch.no_grad():
        our_logits = model(ids)
        hf_logits = hf_model(ids).logits

    # Winzige Float-Unterschiede sind okay; grössere bedeuten ein vergessenes Gewicht
    torch.testing.assert_close(our_logits, hf_logits, atol=1e-4, rtol=1e-4)
    print("GPT-2-Parität BESTANDEN!")

test_gpt2_parity(model, tokenizer)
# Erwartet: GPT-2-Parität BESTANDEN!


In [ ]:
# Text generieren!
prompt = "The quick brown fox"
generated = generate_simple(model, tokenizer, prompt, max_new_tokens=30)

print(f"Prompt: '{prompt}'")
print(f"Generiert: '{generated}'")

In [ ]:
# Mehr Prompts ausprobieren!
prompts = [
    "Artificial intelligence will",
    "Once upon a time",
    "The capital of France is",
    "def fibonacci(n):"
]

for prompt in prompts:
    generated = generate_simple(model, tokenizer, prompt, max_new_tokens=20)
    print(f"'{prompt}' -> {generated}")
    print()

## 7. Vergleich: Zufällige vs. vortrainierte Gewichte

Vergleichen wir eindrucksvoll zufällige Gewichte mit vortrainierten Gewichten.

In [ ]:
# Modell mit zufälligen Gewichten
model_random = MiniGPT(GPTConfig())
model_random = model_random.to(device)

prompt = "Artificial intelligence will"

print("=" * 50)
print("ZUFÄLLIGE GEWICHTE:")
print(generate_simple(model_random, tokenizer, prompt, max_new_tokens=20))
print()
print("VORTRAINIERTE GEWICHTE:")
print(generate_simple(model, tokenizer, prompt, max_new_tokens=20))
print("=" * 50)

print("\nGleiche Architektur. Gleicher Code. Training macht den ganzen Unterschied!")

## 8. Übungen

### Übung 1: Kleines Modell

Baue und teste ein kleines Modell mit spezifischen Spezifikationen.

In [ ]:
# DEIN CODE HIER
# Baue ein kleines MiniGPT mit:
# - vocab_size=500
# - max_seq_len=16
# - embed_dim=64
# - num_heads=2
# - num_layers=2
# - d_ff=256

# 1. Erstelle die Konfiguration
# 2. Baue das Modell
# 3. Führe einen Vorwärtsdurchlauf mit zufälligen Tokens durch
# 4. Überprüfe die Ausgabeform
# 5. Zähle die Parameter

### Übung 2: Temperatur-Exploration

Ändere die Generierung, um Temperaturskalierung zu verwenden.

In [ ]:
# DEIN CODE HIER
# Ändere generate_simple, um einen Temperaturparameter zu akzeptieren:
# next_logits = next_logits / temperature
#
# Probier Temperaturen aus: 0.5, 1.0, 1.5
# Wie ändert sich die Ausgabe?

### Übung 3: Aufmerksamkeitsvisualisierung

Visualisiere Aufmerksamkeitsmuster im vortrainierten Modell.

In [ ]:
# DEIN CODE HIER
# 1. Führe das Modell mit return_attention=True aus
# 2. Hole dir die Aufmerksamkeitsgewichte vom letzten Block
# 3. Zeichne eine Heatmap für Kopf 0
# Tipp: Verwende matplotlib.pyplot.imshow()

### Parametergruppen-Vorschau (in Kapitel 12 verwendet)

Kapitel 12 wendet Weight Decay nur auf die Parameter an, die davon profitieren. Dieser Helfer setzt die Aufteilung um: Matrizen (2D+) bekommen Decay, Biases und LayerNorm-Vektoren nicht.


In [ ]:
def get_parameter_groups(model, weight_decay=0.1):
    """
    Trennt Parameter, die Weight Decay bekommen sollen, von denen ohne.

    Weight Decay ist eine Regularisierung, die kleinere Gewichte fördert.
    Wir wenden sie nicht an auf:
    - LayerNorm-Parameter (sie sollen frei skalieren dürfen)
    - Biases (sie neigen nicht zu Overfitting)
    - Einbettungen sind 2D-Matrizen und bekommen hier Decay (die ndim-Regel)
    """
    decay_params = []
    no_decay_params = []

    for name, param in model.named_parameters():
        # Matrizen (2D+) bekommen Weight Decay; Vektoren (Biases, LayerNorm-Skalen) nicht
        if param.ndim < 2:
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    return [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0}
    ]

# Kapitel 12 baut seinen Optimizer mit diesem Helfer:
# AdamW(get_parameter_groups(model, weight_decay=0.01), lr=learning_rate)
print("get_parameter_groups definiert!")


## Zusammenfassung

**Was wir gebaut haben:**

1. **GPTConfig**: Sauberes Konfigurationsmuster
2. **MiniGPT**: Vollständiges Sprachmodell mit Weight Tying
3. **Sanity Checks**: 6 Tests zur Überprüfung der Korrektheit, inklusive Logits-Parität mit HuggingFace GPT-2
4. **Gewichte laden**: Transfer von GPT-2-Gewichten
5. **Textgenerierung**: Gieriges Dekodieren

**Schlüsselkonzepte:**

- `nn.ModuleList` zum Stapeln von Schichten
- Weight Tying spart 38 Millionen Parameter
- LM-Head: `(batch, seq, embed_dim)` → `(batch, seq, vocab_size)`
- `model.train()` vs. `model.eval()`
- Weight-Tying-Stolperstein: `'lm_head.weight'` erscheint SEHR WOHL in `state_dict().keys()`; gebundene Gewichte teilen sich den Speicher, `torch.save` schreibt ihn nur einmal und die Datei bleibt klein

**Nächster Schritt:** Kapitel 12 zeigt dir, wie du dieses Modell von Grund auf trainierst!